<a href="https://colab.research.google.com/github/Khadija-Azam05/ML-Projects/blob/main/Copy_of_w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Khadija-Azam05/ML-Projects/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — Growing vs declining content

The paper found that growing pages were generally younger and longer than declining pages. Growing pages were around 184 days old on average and had about 3.2K words, while declining pages were around 230 days old and had about 2.3K words. The paper marked this finding as confirmed, but also described it as an observational result.

**My methodology question:** I would like to know whether the 30-day trend label is enough to show a real pattern of growth or decline. A page might have a short-term change in impressions without being consistently in that direction. It would be useful to see whether the same pages continued to grow or decline in a later period.

### Finding 2 — Content performance and age

The paper found that content performance was strongest around 61–90 days and generally became weaker after 270 days. It also found that some older pages performed better when they had been refreshed.

**My methodology question:** I would want to know how much of this difference is actually related to refreshing and how much is simply related to the age of the content. Older pages that were refreshed might already have had better visibility or other advantages. Comparing similar pages over time would make the refresh-related conclusion stronger.

In [ ]:
print("Paper finding 1: growing vs declining content")
print("Growing: 3.2K words, 184 days")
print("Declining: 2.3K words, 230 days")

print("\nPaper finding 2: content performance curve")
print("Peak: 61-90 days")
print("Decline becomes pronounced after 270 days")
print("365+ rebound is concentrated in refreshed older pages")

Paper finding 1: growing vs declining content
Growing: 3.2K words, 184 days
Declining: 2.3K words, 230 days

Paper finding 2: content performance curve
Peak: 61-90 days
Decline becomes pronounced after 270 days
365+ rebound is concentrated in refreshed older pages


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
!git clone https://github.com/Khadija-Azam05/ML-Projects.git

Cloning into 'ML-Projects'...
remote: Enumerating objects: 188, done.
remote: Counting objects: 100% (188/188), done.
remote: Compressing objects: 100% (144/144), done.
remote: Total 188 (delta 79), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (188/188), 1.89 MiB | 14.90 MiB/s, done.
Resolving deltas: 100% (79/79), done.


In [ ]:
import pandas as pd

df = pd.read_csv(
    "ML-Projects/data/raw/content_refresh_anonymized.csv"
)


My Week-5 model used a random stratified 80/20 split and achieved an NDCG@100 score of 0.926767. This split kept the target proportions similar between training and validation, but pages from the same clients could appear in both sets.

For a more honest check, I will now split the data by **client_id**. This means the clients used for validation will not appear in the training data. This gives a better test of whether the model can generalize to clients it did not see during training.

I will compare the grouped result with my Week-5 random-split result. The difference between the two scores is useful evidence about how much the original result may have depended on having the same clients in both training and validation.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ndcg_score
import numpy as np
import pandas as pd

# Features used in Week 5
features = [
    "impressions_90d",
    "clicks_90d",
    "avg_position",
    "days_since_last_update"
]

target = "trend_direction"

# Remove rows with missing values in modeling fields
model_df = df[features + [target, "client_id"]].dropna().copy()

X = model_df[features]
y = model_df[target]
groups = model_df["client_id"]

print("Rows available for modeling:", len(model_df))
print("Unique clients:", groups.nunique())

# --------------------------------------------------
# Honest grouped split
# --------------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_val = X.iloc[val_idx]

y_train = y.iloc[train_idx]
y_val = y.iloc[val_idx]

train_clients = set(groups.iloc[train_idx])
val_clients = set(groups.iloc[val_idx])

print("\nTraining rows:", len(X_train))
print("Validation rows:", len(X_val))

print("\nTraining clients:", len(train_clients))
print("Validation clients:", len(val_clients))

print("\nClients shared between train and validation:")
print(train_clients.intersection(val_clients))

# --------------------------------------------------
# Train Random Forest
# --------------------------------------------------

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf.fit(X_train, y_train)

# --------------------------------------------------
# Validation predictions
# --------------------------------------------------

proba = rf.predict_proba(X_val)

classes = rf.classes_

# NDCG needs relevance scores.
# Use probability of the "down" class,
# because the Week-5 model is being evaluated
# for identifying declining pages.

down_index = list(classes).index("down")

down_scores = proba[:, down_index]

relevance = (y_val == "down").astype(int).to_numpy()

grouped_ndcg = ndcg_score(
    [relevance],
    [down_scores],
    k=min(100, len(relevance))
)

print("\nWeek-5 random split NDCG@100:", 0.926767)
print("ML-09 grouped split NDCG@100:", round(grouped_ndcg, 6))

print(
    "Change from Week-5:",
    round(grouped_ndcg - 0.926767, 6)
)

Rows available for modeling: 30000
Unique clients: 32

Training rows: 23837
Validation rows: 6163

Training clients: 25
Validation clients: 7

Clients shared between train and validation:
set()

Week-5 random split NDCG@100: 0.926767
ML-09 grouped split NDCG@100: 0.769499
Change from Week-5: -0.157268


### Before vs after

My Week-5 random split produced an NDCG@100 of 0.926767. Under the grouped split, the model achieved an NDCG@100 of 0.769499.

The grouped result is lower by 0.157268. This suggests that the Week-5 result was more optimistic when pages from the same clients could appear in both training and validation. The model still shows useful directional ranking performance on clients it did not see during training, but the grouped result is a more cautious estimate of how well the model may generalize.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage audit

I checked the final four modeling features for possible leakage. The target is **trend_direction**, which is derived from **trend_pct**, so I did not use either **trend_direction** or **trend_pct** as model features.

My final features are **impressions_90d**, **clicks_90d**, **avg_position**, and **days_since_last_update**. I also kept **client_id** only for the grouped validation split and did not use it as a model feature.

None of the four selected features is a direct copy of the target or a target-derived column. However, the 90-day features need careful interpretation because the target is based on trend information over a time window. Without a precise prediction date and feature/label window construction, I cannot claim that the starter CSV proves there is no temporal overlap. I therefore treat the grouped result as a more cautious validation result rather than claiming that leakage has been completely ruled out.

In [ ]:
# Final feature leakage check

final_features = [
    "impressions_90d",
    "clicks_90d",
    "avg_position",
    "days_since_last_update"
]

target = "trend_direction"

forbidden_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

print("Final model features:")
print(final_features)

print("\nTarget:")
print(target)

print("\nForbidden / target-derived columns:")
print(forbidden_columns)

print("\nTarget-derived columns accidentally used as features:")

leaky_features = [
    col for col in final_features
    if col in forbidden_columns
]

if leaky_features:
    print(leaky_features)
else:
    print("None")

print("\nClient ID used as model feature?")
print("client_id" in final_features)

Final model features:
['impressions_90d', 'clicks_90d', 'avg_position', 'days_since_last_update']

Target:
trend_direction

Forbidden / target-derived columns:
['trend_direction', 'trend_pct', 'is_declining_label']

Target-derived columns accidentally used as features:
None

Client ID used as model feature?
False


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original claim

The Random Forest model significantly improves the baseline and can identify declining content pages effectively.

### Safer claim

The Random Forest produced a higher NDCG@100 than the Week-4 baseline on the Week-5 random split, with a score of 0.926767 compared with 0.475782. However, when I evaluated the same model using a client-grouped split, the score decreased to 0.769499. This suggests that the random-split result was more optimistic, so the model should be treated as a directional, decision-support tool for ranking potential declining content rather than as proof that it will perform equally well for unseen clients.

In [ ]:
print("Week-4 baseline NDCG@100:", 0.475782)
print("Week-5 random-split NDCG@100:", 0.926767)
print("ML-09 grouped-split NDCG@100:", 0.769499)

print("\nLeakage audit:")
print("Direct label-derived features used: None")
print("Client ID used as model feature: False")
print("Temporal overlap fully verifiable from starter CSV: False")

Week-4 baseline NDCG@100: 0.475782
Week-5 random-split NDCG@100: 0.926767
ML-09 grouped-split NDCG@100: 0.769499

Leakage audit:
Direct label-derived features used: None
Client ID used as model feature: False
Temporal overlap fully verifiable from starter CSV: False
